### Mesh loading and a nonlinear PDE

In this exercise, we will load a mesh and solve minimal surface problem describe by:
\begin{align*}
    -\nabla \cdot \left( \frac{\nabla u}{\sqrt{1 + |\nabla u|^2}} \right) &= -1 &&\text{ in } \Omega \\
    u &= 0 &&\text{ on } \Gamma_D \\
    - \nabla u \cdot n &= 0 &&\text{ on } \Gamma_N
\end{align*}
The equation describes a surface under a gravitational force. In this problem the boundary splits into Dirichlet and Neumann parts, denoted by $\Gamma_D$ and $\Gamma_N$, respectively.

The domain is given by a mesh that we will load in the following steps.

In [ ]:
# # This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

path = "/localdata/tomfre/SolutionData/MinimalSurface/"

As usual we define the variables first, including the spatial coordinates, the solution, and the normal vector.

In [ ]:
import qewton

X = qewton.Variable("x", 2)
U = qewton.Variable() # TODO: Add a variable for the solution
N = qewton.Variable("n", dim=) # TODO: Add a variable for the normal vector (dim=2)

Next we load the mesh from a file located at the specified path.

In [ ]:
omega = qewton.geometries.MeshGeometry.load_mesh(
    variable=X, 
    file_path=path + "unit_square_minus_circle.msh",
    prune_z=True # Its a 2D problem, so we prune the z-coordinate
)

# We can extract the submeshes corresponding to the Dirichlet and Neumann boundaries.
gamma_D = omega.boundary.get_submesh("dirichlet")
# TODO: Extract the Neumann boundary submesh, named "neumann"
gamma_N = omega.boundary.get_submesh() 

We can have a look at the loaded mesh:

In [ ]:
qewton.viz.Figure(qewton.viz.GeometryPlot(omega)).show()

Internally the loaded mesh can be used like any other geometry object in Qewton.
Next, we create the samplers for the domain and the boundaries.

In [ ]:
inner_sampler = qewton.GridSampler(omega, n_points=25000)
# TODO: Create a sampler for the Dirichlet and Neumann boundary
gamma_D_sampler = qewton.GridSampler(, n_points=8000)
gamma_N_sampler = qewton.GridSampler(, n_points=8000, normal_name=N, compute_normals=True)
# For the Neumann boundary, we need to compute the normals.

In [ ]:
# TODO: Build the neural network model for the PDE solution, 
model = qewton.FCN(...)

Next up are the residuals for the PDE and the boundary conditions.
We start with the PDE residual:

In [ ]:
def pde_residual(u : U, x : X):
    u_grad = # TODO: Compute the gradient of u with respect to x
    u_grad_norm = u_grad.norm()
    scaling = ().sqrt() # TODO: Implement the denominator from the PDE
    return - (u_grad / scaling).div(x) + 1.0

pde_constraint = qewton.constraints.PINNConstraint(pde_residual)

# TODO: Build the PINN pipeline for the PDE constraint
pde_pipeline = qewton.PINNPipeline(
    
)

In [ ]:
# TODO: Define the Dirichlet boundary condition residual
def dirichlet_residual():

dirichlet_constraint = qewton.constraints.PINNConstraint(
    dirichlet_residual, 
    weight=10.0, 
    name="DirichletConstraint"
)
dirichlet_pipeline = qewton.PINNPipeline(gamma_D_sampler, [model], dirichlet_constraint)

In [ ]:
# TODO: Define the Neumann boundary condition residual and constraint
# For the normal derivative in the Neumann boundary condition use
# u.normal_derivative(n, x) which
# "computes the normal derivative of u at x in the direction of n"
def neumann_residual(u: U, x: X, n: N):
    return 

neuman_constraint = qewton.constraints.PINNConstraint(
    neumann_residual, 
    weight=10.0, 
    name="NeumannConstraint"
)
neumann_pipeline = qewton.PINNPipeline(gamma_N_sampler, [model], neuman_constraint)

Now we start training. But observe, that we use two separate optimizers in the training. We start
with Adam to first pre-train the model on the PDE and boundary condition constraints. Then we fine
tune with LBFGS a higher order optimizer for better convergence.

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=1.0,
    max_iterations=100,
    optimizer_args={"max_eval": 5}
)

# TODO: Collect all the graphs and training objectives
graphs = []
training_objectives = []

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=graphs,
    training_objectives=training_objectives,
    device=qewton.cuda(0)
)

trainer.run()

Finally we plot the solution and compare it with an FEM reference solution.

In [ ]:
solution_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "solution.pt")
reference_config = qewton.DataConfiguration(
    qewton.GeometryAxes(geometry=omega), qewton.FeatureAxes(U)
)

layout = pde_pipeline.visualize(model.output_ports[0], 
                             reference=solution_data, 
                             reference_config=reference_config)
qewton.viz.Figure(layout).show()